## Importing Required Libraries and Loading CSV's 

In [ ]:
#importing libraries and loading csv's
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats

#loading all 9 csv
c_d = pd.read_csv('CSV Files/olist_customers_dataset.csv')
g_d = pd.read_csv('CSV Files/olist_geolocation_dataset.csv')
o_i_d = pd.read_csv('CSV Files/olist_order_items_dataset.csv')
o_p_d = pd.read_csv('CSV Files/olist_order_payments_dataset.csv')
o_r_d = pd.read_csv('CSV Files/olist_order_reviews_dataset.csv')
o_d = pd.read_csv('CSV Files/olist_orders_dataset.csv')
p_d = pd.read_csv('CSV Files/olist_products_dataset.csv')
s_d = pd.read_csv('CSV Files/olist_sellers_dataset.csv')
p_c_n_t = pd.read_csv('CSV Files/product_category_name_translation.csv') 

## Data EDA phase 

In [17]:
temp_dict = {
    'olist_customers_dataset': c_d,
    'olist_geolocation_dataset': g_d,
    'olist_order_items_dataset': o_i_d,
    'olist_order_payments_dataset': o_p_d,
    'olist_order_reviews_dataset': o_r_d,
    'olist_orders_dataset': o_d,
    'olist_products_dataset': p_d,
    'olist_sellers_dataset': s_d,
    'product_category_name_translation': p_c_n_t
}
for i in temp_dict:
    print(f"\n----{i}----")
    print(f"{i} shape: {temp_dict[i].shape}")
    print(f"\n{temp_dict[i].isnull().sum()}")


----olist_customers_dataset----
olist_customers_dataset shape: (99441, 5)

customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64

----olist_geolocation_dataset----
olist_geolocation_dataset shape: (1000163, 5)

geolocation_zip_code_prefix    0
geolocation_lat                0
geolocation_lng                0
geolocation_city               0
geolocation_state              0
dtype: int64

----olist_order_items_dataset----
olist_order_items_dataset shape: (112650, 7)

order_id               0
order_item_id          0
product_id             0
seller_id              0
shipping_limit_date    0
price                  0
freight_value          0
dtype: int64

----olist_order_payments_dataset----
olist_order_payments_dataset shape: (103886, 5)

order_id                0
payment_sequential      0
payment_type            0
payment_installments    0
payment_value           0
dtype: int64

-

## Data Cleaning 

In [52]:
# checking order status and then deleting required values
print(temp_dict['olist_orders_dataset']['order_status'].value_counts())
clean_values = o_d[~o_d['order_status'].isin(['canceled', 'unavailable'])].copy()
print(clean_values['order_status'].value_counts())
print(clean_values.shape)

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64
order_status
delivered     96478
shipped        1107
invoiced        314
processing      301
created           5
approved          2
Name: count, dtype: int64
(98207, 8)


In [53]:
# droppping off the required columns 
clean_values = clean_values[~((clean_values['order_status'] == 'delivered')&(clean_values['order_delivered_customer_date'].isnull()))]
print(clean_values['order_status'].value_counts())
print(clean_values.shape)

order_status
delivered     96470
shipped        1107
invoiced        314
processing      301
created           5
approved          2
Name: count, dtype: int64
(98199, 8)


## Converting to date columns to datetime

In [54]:
## olists_order_dataset
date_columns = ['order_purchase_timestamp',
                'order_approved_at',
                'order_delivered_carrier_date',
                'order_delivered_customer_date',
                'order_estimated_delivery_date']
for col in date_columns:
    clean_values[col] = pd.to_datetime(clean_values[col])
print(clean_values.dtypes)

order_id                                 object
customer_id                              object
order_status                             object
order_purchase_timestamp         datetime64[ns]
order_approved_at                datetime64[ns]
order_delivered_carrier_date     datetime64[ns]
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
dtype: object


In [56]:
#creating df for actual delivery date 
clean_values['actual_delivery_date'] = (clean_values['order_delivered_customer_date'] - clean_values['order_estimated_delivery_date']).dt.days
print(clean_values['actual_delivery_date'].head(10))

0    -8.0
1    -6.0
2   -18.0
3   -13.0
4   -10.0
5    -6.0
6     NaN
7   -12.0
8   -32.0
9    -7.0
Name: actual_delivery_date, dtype: float64


In [57]:
# merging product category name 
merged_df = pd.merge(p_d, p_c_n_t, on= "product_category_name")
print(merged_df.head())

                         product_id  product_category_name  \
0  1e9e8ef04dbcff4541ed26657ea517e5             perfumaria   
1  3aa071139cb16b67ca9e5dea641aaa2f                  artes   
2  96bd76ec8810374ed1b65e291975717f          esporte_lazer   
3  cef67bcfe19066a932b7673e239eb23d                  bebes   
4  9dc1a7de274444849c219cff195d0b71  utilidades_domesticas   

   product_name_lenght  product_description_lenght  product_photos_qty  \
0                 40.0                       287.0                 1.0   
1                 44.0                       276.0                 1.0   
2                 46.0                       250.0                 1.0   
3                 27.0                       261.0                 1.0   
4                 37.0                       402.0                 4.0   

   product_weight_g  product_length_cm  product_height_cm  product_width_cm  \
0             225.0               16.0               10.0              14.0   
1            1000.0     

In [1]:
#merging required data
# orders → items (links to products & sellers)
master_df = pd.merge(clean_values, o_i_d, on='order_id', how='left')

# products (with english category names)
master_df = pd.merge(master_df, merged_df, on='product_id', how='left')

# sellers
master_df = pd.merge(master_df, s_d, on='seller_id', how='left')

# customers
master_df = pd.merge(master_df, c_d, on='customer_id', how='left')

# payments
master_df = pd.merge(master_df, o_p_d, on='order_id', how='left')

# reviews
master_df = pd.merge(master_df, o_r_d, on='order_id', how='left')

print(master_df.shape)
print(master_df.columns.tolist())

NameError: name 'pd' is not defined